# 3-Year 95% CVaR with a D-vine Copula
This notebook implements the requested pipeline for quarterly returns using `polars` for data handling and simulation, `altair` for visualizations, and a D-vine copula for cross-sectional dependence.

Pipeline:
1. Load the provided quarterly return dataset.
2. Optionally unsmooth private-asset returns if you are starting from appraisal-smoothed series.
3. Fit ARMA(1,1)-GARCH(1,1) marginals with either skewed-t or stable standardized residuals.
4. Fit a D-vine copula to the PIT residuals.
5. Simulate 3-year quarterly paths and estimate 95% CVaR.

If `pyvinecopulib` is not installed yet, install it before running this notebook.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import warnings

import altair as alt
import numpy as np
import pandas as pd
import polars as pl
from arch import arch_model
from scipy.stats import levy_stable, norm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import LinearRegression

try:
    import pyvinecopulib as pv
except ImportError as exc:
    pv = None
    PYVINECOPULIB_IMPORT_ERROR = exc

warnings.filterwarnings('ignore')
alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', None)

DATA_PATH = Path('../data/processed_returns.csv')
DATE_COLUMN = 'date'
HORIZON_QUARTERS = 12
N_SIMULATIONS = 50_000
ALPHA = 0.05
RESIDUAL_FAMILY = 'normal'  # set to 'stable' to use a stable residual distribution
APPLY_UNSMOOTHING = False  # keep False for the provided processed dataset
PRIVATE_ASSETS = {'PE', 'NPI'}
PORTFOLIO_WEIGHTS = {'SPY': 0.40, 'AGG': 0.20, 'PE': 0.25, 'NPI': 0.15}
EXPECTED_RETURNS = {'SPY': (1+0.06)**(1/4) - 1, 'AGG': (1+0.045)**(1/4) - 1, 'PE': (1+0.095)**(1/4) - 1, 'NPI': (1+0.07)**(1/4) - 1}
D_VINE_ORDER = ['SPY', 'AGG', 'PE', 'NPI']
RNG_SEED = 7

print('Configuration loaded.')

Configuration loaded.


In [2]:
ret_df = pd.read_csv('../data/processed_returns.csv', index_col=0, parse_dates=True)
ret_df = ret_df/100
ret_df = np.log(1 + ret_df)
ret_df.index.name = DATE_COLUMN
asset_cols = [column for column in D_VINE_ORDER if column in ret_df.columns]
if len(asset_cols) != 4:
    asset_cols = [column for column in ret_df.columns if pd.api.types.is_numeric_dtype(ret_df[column])]
    asset_cols = list(asset_cols)

if APPLY_UNSMOOTHING:
    print('Applying Geltner-style unsmoothing to private assets.')
    for asset in PRIVATE_ASSETS.intersection(asset_cols):
        values = ret_df[asset].to_numpy(dtype=float)
        if len(values) > 1:
            model = LinearRegression().fit(values[:-1].reshape(-1, 1), values[1:])
            alpha = model.coef_[0]
            alpha = float(np.clip(alpha, 0.0, 0.95))
            unsmoothed = values.copy()
            unsmoothed[0] = values[0]
            
            unsmoothed[1:] = (values[1:] - alpha * values[:-1]) / (1.0 - alpha)
            
            smoothed_ann_vol = np.std(values) * np.sqrt(4)
            unsmoothed_ann_vol = np.std(unsmoothed) * np.sqrt(4)
            
            print(f'Asset: {asset}, alpha: {alpha:.4f}, Smoothed Annual Vol: {smoothed_ann_vol:.4f}, Unsmoothed Annual Vol: {unsmoothed_ann_vol:.4f}')
            
            ret_df[asset] = unsmoothed

model_df = ret_df[asset_cols].dropna().copy()
ret_pl = pl.from_pandas(model_df.reset_index())
returns_long = ret_pl.unpivot(index=DATE_COLUMN, on=asset_cols, variable_name='asset', value_name='return')

print(f'Raw shape: {ret_df.shape}')
print(f'Modeling shape: {model_df.shape}')
print('Assets:', asset_cols)
ret_pl.head()

Raw shape: (96, 4)
Modeling shape: (81, 4)
Assets: ['SPY', 'AGG', 'PE', 'NPI']


date,SPY,AGG,PE,NPI
datetime[μs],f64,f64,f64,f64
2003-12-31 00:00:00,0.112094,0.003692,0.094241,0.034574
2004-03-31 00:00:00,0.019722,0.02256,-0.044602,0.092938
2004-06-30 00:00:00,0.016208,-0.024996,0.076009,0.062427
2004-09-30 00:00:00,-0.020329,0.030419,0.064438,0.060961
2004-12-31 00:00:00,0.086034,0.009152,0.039644,0.049436


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import t as student_t
from scipy.special import gammaln
from dataclasses import dataclass
from typing import Optional


# ── DATA CLASS FOR PARAMETERS ────────────────────────────────────────────────

@dataclass
class GARCHParams:
    mu:        float          # constant mean
    phi:       float          # AR(1) coefficient
    theta:     float          # MA(1) coefficient
    alpha:     float          # ARCH coefficient
    beta:      float          # GARCH coefficient
    omega:     float          # variance intercept (variance-targeted)
    sigma2_lr: float          # long-run variance (sample variance)
    nu:        Optional[float] = None   # degrees of freedom (t / skew-t)
    lam:       Optional[float] = None   # skewness parameter (skew-t only)

    def summary(self, annualise_factor: int = 4):
        print("═" * 45)
        print("  ARMA(1,1)-GARCH(1,1) Parameter Estimates")
        print("═" * 45)
        print(f"  Mean equation")
        print(f"    mu           = {self.mu:.6f}")
        print(f"    phi (AR1)    = {self.phi:.6f}")
        print(f"    theta (MA1)  = {self.theta:.6f}")
        print(f"  Variance equation")
        print(f"    omega        = {self.omega:.6f}  (variance-targeted)")
        print(f"    alpha        = {self.alpha:.6f}")
        print(f"    beta         = {self.beta:.6f}")
        print(f"    alpha+beta   = {self.alpha + self.beta:.6f}  "
              f"({'✓ stationary' if self.alpha + self.beta < 1 else '✗ NON-STATIONARY'})")
        print(f"  Long-run vol   = "
              f"{np.sqrt(self.sigma2_lr * annualise_factor) * 100:.2f}% (annualised)")
        if self.nu is not None:
            print(f"  Tail (nu)      = {self.nu:.4f}")
        if self.lam is not None:
            print(f"  Skew (lambda)  = {self.lam:.4f}")
        print("═" * 45)


# ── LOG-LIKELIHOOD FUNCTIONS ─────────────────────────────────────────────────

def _garch_variance_path(resid: np.ndarray, omega: float,
                         alpha: float, beta: float,
                         sigma2_lr: float) -> np.ndarray:
    """
    Compute the GARCH(1,1) conditional variance path.
    Initialised at the long-run variance (variance targeting start condition).
    """
    n = len(resid)
    sigma2 = np.empty(n)
    sigma2[0] = sigma2_lr                          # initialise at long-run variance
    for t in range(1, n):
        sigma2[t] = omega + alpha * resid[t-1]**2 + beta * sigma2[t-1]
        sigma2[t] = max(sigma2[t], 1e-8)           # floor for numerical safety
    return sigma2


def _residuals(r: np.ndarray, mu: float,
               phi: float, theta: float) -> np.ndarray:
    """
    Compute ARMA(1,1) residuals:  eps_t = r_t - mu - phi*r_{t-1} - theta*eps_{t-1}
    """
    n = len(r)
    eps = np.zeros(n)
    for t in range(1, n):
        eps[t] = r[t] - mu - phi * r[t-1] - theta * eps[t-1]
    return eps


def _nll_normal(params: np.ndarray, r: np.ndarray, sigma2_lr: float) -> float:
    """Gaussian ARMA-GARCH negative log-likelihood with variance targeting."""
    mu, phi, theta, alpha, beta = params

    # Stationarity and positivity guards — return large penalty if violated
    if not (abs(phi) < 1 and abs(theta) < 1):      return 1e10
    if not (alpha > 0 and beta > 0):                return 1e10
    if alpha + beta >= 0.9999:                      return 1e10

    omega = sigma2_lr * (1 - alpha - beta)          # variance targeting
    eps   = _residuals(r, mu, phi, theta)
    sig2  = _garch_variance_path(eps, omega, alpha, beta, sigma2_lr)

    # Gaussian log-likelihood: -0.5 * sum[ log(2pi) + log(sig2) + eps^2/sig2 ]
    nll = 0.5 * np.sum(np.log(2 * np.pi) + np.log(sig2) + eps**2 / sig2)
    return nll


def _nll_student_t(params: np.ndarray, r: np.ndarray, sigma2_lr: float) -> float:
    """Student-t ARMA-GARCH negative log-likelihood with variance targeting."""
    mu, phi, theta, alpha, beta, nu = params

    if not (abs(phi) < 1 and abs(theta) < 1):      return 1e10
    if not (alpha > 0 and beta > 0):                return 1e10
    if alpha + beta >= 0.9999:                      return 1e10
    if nu <= 2.01:                                  return 1e10   # variance must exist

    omega = sigma2_lr * (1 - alpha - beta)
    eps   = _residuals(r, mu, phi, theta)
    sig2  = _garch_variance_path(eps, omega, alpha, beta, sigma2_lr)
    sigma = np.sqrt(sig2)

    # Standardised t log-likelihood
    # Var(t_nu) = nu/(nu-2), so standardise innovations by sqrt(nu/(nu-2))
    scale = np.sqrt(nu / (nu - 2))
    z     = eps / (sigma * scale)      # z ~ t(nu) with unit variance
    
    log_const = gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2))
    log_kern  = -((nu + 1) / 2) * np.log(1 + z**2 / (nu - 2))
    log_jacob = -np.log(sigma * scale)  # jacobian for sigma scaling

    nll = -np.sum(log_const + log_kern + log_jacob)
    return nll


def _nll_skewt(params: np.ndarray, r: np.ndarray, sigma2_lr: float) -> float:
    """
    Hansen (1994) Skewed-t ARMA-GARCH negative log-likelihood.
    Captures both fat tails (nu) and asymmetry (lam in (-1, 1)).
    """
    mu, phi, theta, alpha, beta, nu, lam = params

    if not (abs(phi) < 1 and abs(theta) < 1):       return 1e10
    if not (alpha > 0 and beta > 0):                 return 1e10
    if alpha + beta >= 0.9999:                       return 1e10
    if nu <= 2.01:                                   return 1e10
    if not (-0.999 < lam < 0.999):                   return 1e10

    omega = sigma2_lr * (1 - alpha - beta)
    eps   = _residuals(r, mu, phi, theta)
    sig2  = _garch_variance_path(eps, omega, alpha, beta, sigma2_lr)
    sigma = np.sqrt(sig2)
    z     = eps / sigma

    # Hansen (1994) constants
    c = np.exp(gammaln((nu + 1) / 2) - gammaln(nu / 2)) / np.sqrt(np.pi * (nu - 2))
    a = 4 * lam * c * ((nu - 2) / (nu - 1))
    b = np.sqrt(1 + 3 * lam**2 - a**2)

    # Split domain: f(z) depends on sign of b*z + a
    bza = b * z + a
    ll  = np.empty(len(z))

    pos = bza >= 0
    # Right tail (z >= -a/b)
    ll[pos]  = (np.log(b) + np.log(c) - np.log(sigma[pos])
                - ((nu + 1) / 2)
                * np.log(1 + (bza[pos] / (1 + lam)) ** 2 / (nu - 2)))
    # Left tail (z < -a/b)
    ll[~pos] = (np.log(b) + np.log(c) - np.log(sigma[~pos])
                - ((nu + 1) / 2)
                * np.log(1 + (bza[~pos] / (1 - lam)) ** 2 / (nu - 2)))

    return -np.sum(ll)


# ── MAIN FITTING FUNCTION ────────────────────────────────────────────────────

def fit_garch(
    returns:   pd.Series,
    dist:      str = 'skewt',
    n_restarts: int = 5,
    seed:      int = 42
) -> GARCHParams:
    """
    Fit ARMA(1,1)-GARCH(1,1) with variance targeting via direct MLE.
    No arch library — pure numpy/scipy.

    Parameters
    ----------
    returns : pd.Series
        Return series (quarterly decimals, e.g. 0.05 = 5%)
    dist : str
        'normal', 't', or 'skewt'
    n_restarts : int
        Number of random restarts to escape local optima
    seed : int

    Returns
    -------
    GARCHParams dataclass
    """
    rng      = np.random.default_rng(seed)
    r        = returns.values.astype(float)
    sigma2_lr = r.var()

    dist_map = {
        'normal': (_nll_normal,    ['mu','phi','theta','alpha','beta']),
        't':      (_nll_student_t, ['mu','phi','theta','alpha','beta','nu']),
        'skewt':  (_nll_skewt,     ['mu','phi','theta','alpha','beta','nu','lam']),
    }
    if dist not in dist_map:
        raise ValueError(f"dist must be one of {list(dist_map.keys())}")

    nll_fn, param_names = dist_map[dist]

    # ── Starting values ──────────────────────────────────────────────────────
    def make_x0(random=False):
        if random:
            phi_0   = rng.uniform(-0.3, 0.3)
            theta_0 = rng.uniform(-0.3, 0.3)
            ab      = rng.uniform(0.70, 0.95)
            alpha_0 = rng.uniform(0.03, 0.25)
            beta_0  = ab - alpha_0
        else:
            phi_0, theta_0, alpha_0, beta_0 = 0.1, 0.05, 0.10, 0.85

        base = [r.mean(), phi_0, theta_0, alpha_0, beta_0]
        if dist == 't':     return base + [8.0]
        if dist == 'skewt': return base + [8.0, -0.1]
        return base

    # ── Bounds ───────────────────────────────────────────────────────────────
    base_bounds = [
        (-0.2, 0.2),    # mu
        (-0.95, 0.95),  # phi
        (-0.95, 0.95),  # theta
        (1e-6, 0.45),   # alpha
        (1e-6, 0.9999), # beta
    ]
    if dist == 't':     bounds = base_bounds + [(2.1, 50.0)]
    elif dist == 'skewt': bounds = base_bounds + [(2.1, 50.0), (-0.999, 0.999)]
    else:               bounds = base_bounds

    # ── alpha + beta < 1 constraint ──────────────────────────────────────────
    constraints = [{
        'type': 'ineq',
        'fun':  lambda x: 0.9999 - (x[3] + x[4])   # alpha + beta < 0.9999
    }]

    # ── Multi-start optimisation ─────────────────────────────────────────────
    best_result = None
    best_nll    = np.inf

    for i in range(n_restarts):
        x0 = make_x0(random=(i > 0))
        try:
            res = minimize(
                nll_fn,
                x0,
                args=(r, sigma2_lr),
                method='SLSQP',
                bounds=bounds,
                constraints=constraints,
                options={'ftol': 1e-10, 'maxiter': 2000, 'disp': False}
            )
            if res.success and res.fun < best_nll:
                best_nll    = res.fun
                best_result = res
        except Exception:
            continue

    if best_result is None:
        raise RuntimeError("All optimisation restarts failed. "
                           "Check your return series for NaNs or extreme outliers.")

    x = best_result.x
    alpha, beta = x[3], x[4]
    omega = sigma2_lr * (1 - alpha - beta)

    params = GARCHParams(
        mu=x[0], phi=x[1], theta=x[2],
        alpha=alpha, beta=beta, omega=omega,
        sigma2_lr=sigma2_lr,
        nu=x[5]  if dist in ('t', 'skewt') else None,
        lam=x[6] if dist == 'skewt'        else None,
    )

    print(f"Converged in {best_result.nit} iterations  |  "
          f"NLL = {best_nll:.4f}  |  dist = {dist}")
    params.summary()
    return params


# ── INFORMATION CRITERIA ─────────────────────────────────────────────────────

def model_aic_bic(nll: float, n_params: int, n_obs: int) -> dict:
    aic = 2 * nll + 2 * n_params
    bic = 2 * nll + n_params * np.log(n_obs)
    return {'AIC': aic, 'BIC': bic, 'NLL': nll}


def select_distribution(returns: pd.Series, n_restarts: int = 5) -> str:
    """Fit all three distributions and select by BIC."""
    r = returns.values
    n = len(r)
    results = {}

    n_params_map = {'normal': 5, 't': 6, 'skewt': 7}

    print("Comparing distributions by BIC...\n")
    for dist in ['normal', 't', 'skewt']:
        try:
            params = fit_garch(returns, dist=dist, n_restarts=n_restarts)
            nll_fn = {'normal': _nll_normal,
                      't':      _nll_student_t,
                      'skewt':  _nll_skewt}[dist]
            x = [params.mu, params.phi, params.theta, params.alpha, params.beta]
            if params.nu  is not None: x.append(params.nu)
            if params.lam is not None: x.append(params.lam)
            nll = nll_fn(np.array(x), r, params.sigma2_lr)
            ic  = model_aic_bic(nll, n_params_map[dist], n)
            results[dist] = (params, ic)
            print(f"  {dist:<8}  AIC={ic['AIC']:.2f}  BIC={ic['BIC']:.2f}\n")
        except Exception as e:
            print(f"  {dist} failed: {e}")

    best = min(results, key=lambda d: results[d][1]['BIC'])
    print(f"  → Best distribution by BIC: {best.upper()}")
    return best, results[best][0]


In [148]:
# Fit a single distribution
params = fit_garch(ret_df["AGG"].dropna(), dist='skewt', n_restarts=5)

Converged in 25 iterations  |  NLL = -196.4981  |  dist = skewt
═════════════════════════════════════════════
  ARMA(1,1)-GARCH(1,1) Parameter Estimates
═════════════════════════════════════════════
  Mean equation
    mu           = 0.016502
    phi (AR1)    = -0.895511
    theta (MA1)  = 0.950000
  Variance equation
    omega        = 0.000043  (variance-targeted)
    alpha        = 0.086203
    beta         = 0.831195
    alpha+beta   = 0.917399  (✓ stationary)
  Long-run vol   = 4.56% (annualised)
  Tail (nu)      = 6.8272
  Skew (lambda)  = -0.1706
═════════════════════════════════════════════


In [3]:
def fit_arma_garch_marginal(series: pd.Series, residual_family: str = 'skewt') -> dict:
    values = series.dropna().astype(float).to_numpy()
    if values.size < 12:
        raise ValueError(f'{series.name} has too few observations for ARMA-GARCH fitting.')

    garch_dist = residual_family
    garch_fit = arch_model(
        values, # original: arma_resid
        mean='AR', # change to Z to AR
        lags=1,
        vol='GARCH',
        p=1,
        q=1,
        dist=garch_dist,
        rescale=False,
    ).fit(options={'ftol': 1e-9})

    std_resid_raw = garch_fit.std_resid
    if hasattr(std_resid_raw, 'dropna'):
        std_resid_raw = std_resid_raw.dropna()
    std_resid = np.asarray(std_resid_raw, dtype=float)
    sigma = garch_fit.conditional_volatility
    print(f'{series.name} - ARMA-GARCH fitted. Residuals: {std_resid*sigma}')
    eps = 1e-10

    if residual_family == 'skewt':
        from arch.univariate import SkewStudent

        dist = SkewStudent()
        eta = float(garch_fit.params['eta'])
        lam = float(garch_fit.params['lambda'])
        dist_params = np.array([eta, lam], dtype=float)
        u = np.asarray(dist.cdf(std_resid, dist_params), dtype=float)

        def ppf_fn(q: np.ndarray) -> np.ndarray:
            q = np.clip(np.asarray(q, dtype=float), eps, 1.0 - eps)
            return np.asarray(dist.ppf(q, dist_params), dtype=float)

        dist_summary = {'family': 'skewt', 'eta': eta, 'lambda': lam}
        
    elif residual_family == 't':
        from arch.univariate import StudentsT

        dist = StudentsT()
        nu = float(garch_fit.params['nu'])
        dist_params = np.array([nu], dtype=float)
        u = np.asarray(dist.cdf(std_resid, dist_params), dtype=float)

        def ppf_fn(q: np.ndarray) -> np.ndarray:
            q = np.clip(np.asarray(q, dtype=float), eps, 1.0 - eps)
            return np.asarray(dist.ppf(q, dist_params), dtype=float)

        dist_summary = {'family': 't', 'nu': nu}
        
    elif residual_family == 'normal':
        normal_params = norm.fit(std_resid[~np.isnan(std_resid)])
        u = np.asarray(norm.cdf(std_resid[~np.isnan(std_resid)], *normal_params), dtype=float)

        def ppf_fn(q: np.ndarray) -> np.ndarray:
            q = np.clip(np.asarray(q, dtype=float), eps, 1.0 - eps)
            return np.asarray(norm.ppf(q, *normal_params), dtype=float)

        dist_summary = {'family': 'normal'}
    
    elif residual_family == 'stable':
        stable_params = levy_stable.fit(std_resid)
        u = np.asarray(levy_stable.cdf(std_resid, *stable_params), dtype=float)

        def ppf_fn(q: np.ndarray) -> np.ndarray:
            q = np.clip(np.asarray(q, dtype=float), eps, 1.0 - eps)
            return np.asarray(levy_stable.ppf(q, *stable_params), dtype=float)

        dist_summary = {
            'family': 'stable',
            'stable_alpha': float(stable_params[0]),
            'stable_beta': float(stable_params[1]),
            'stable_loc': float(stable_params[2]),
            'stable_scale': float(stable_params[3]),
        }
    else:
        raise ValueError('residual_family must be either skewt, t, normal, or stable.')

    u = np.clip(u, eps, 1.0 - eps)

    params = garch_fit.params
    conditional_volatility = np.asarray(garch_fit.conditional_volatility, dtype=float)
    model_summary = {
        'Asset': series.name,
        'N': int(values.size),
        'GARCH_AIC': float(garch_fit.aic),
        'GARCH_BIC': float(garch_fit.bic),
        'Const':float(params.get('Const', np.nan)),
        'Phi':float(params.get('y[1]', np.nan)),
        'Omega': float(params.get('omega', np.nan)),
        'Alpha': float(params.get('alpha[1]', np.nan)),
        'Beta': float(params.get('beta[1]', np.nan)),
        'Persistence': float(params.get('alpha[1]', 0.0) + params.get('beta[1]', 0.0)),
        'sigma_L': float(params.get('omega', np.nan)) / (1.0 - float(params.get('alpha[1]', 0.0)) - float(params.get('beta[1]', 0.0))) if (1.0 - float(params.get('alpha[1]', 0.0)) - float(params.get('beta[1]', 0.0))) > 0 else np.nan,
    }
    model_summary.update(dist_summary)

    return {
        'asset': series.name,
        'observed': values,
        'garch_fit': garch_fit,
        'std_resid': std_resid,
        'sigma': conditional_volatility,
        'last_std_resid': float(std_resid[-1]),
        'pit': u,
        'ppf': ppf_fn,
        'model_summary': model_summary,
        'last_return': float(values[-1]),
        'last_sigma': float(conditional_volatility[-1]),
        'const':float(params.get('Const', np.nan)),
        'phi':float(params.get('y[1]', np.nan)),
        'omega': float(params.get('omega', np.nan)),
        'alpha': float(params.get('alpha[1]', 0.0)),
        'beta': float(params.get('beta[1]', 0.0)),
    }


marginals = [fit_arma_garch_marginal(model_df[column], residual_family=RESIDUAL_FAMILY) for column in asset_cols]
marginal_summary = pl.DataFrame([fit['model_summary'] for fit in marginals])
marginal_summary

Iteration:      1,   Func. Count:      7,   Neg. LLF: 26861.168871774058
Iteration:      2,   Func. Count:     18,   Neg. LLF: 61.75253394340691
Iteration:      3,   Func. Count:     28,   Neg. LLF: 988275960.130666
Iteration:      4,   Func. Count:     37,   Neg. LLF: -78.8337534636606
Iteration:      5,   Func. Count:     44,   Neg. LLF: -94.21874514841319
Iteration:      6,   Func. Count:     50,   Neg. LLF: -69.65431224991383
Iteration:      7,   Func. Count:     58,   Neg. LLF: -94.28181880676831
Iteration:      8,   Func. Count:     64,   Neg. LLF: -94.29127805852688
Iteration:      9,   Func. Count:     70,   Neg. LLF: -94.29225734477083
Iteration:     10,   Func. Count:     76,   Neg. LLF: -94.29232118675907
Iteration:     11,   Func. Count:     82,   Neg. LLF: -94.29232958467793
Iteration:     12,   Func. Count:     88,   Neg. LLF: -94.29232963596762
Iteration:     13,   Func. Count:     94,   Neg. LLF: -94.29232964043844
Iteration:     14,   Func. Count:     99,   Neg. LLF: -

Asset,N,GARCH_AIC,GARCH_BIC,Const,Phi,Omega,Alpha,Beta,Persistence,sigma_L,family
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""SPY""",81,-178.584659,-166.674526,0.021302,-0.043817,0.000574,0.387363,0.612107,0.99947,1.08318,"""normal"""
"""AGG""",81,-371.981413,-360.07128,0.008285,0.00056,0.000056,0.097754,0.808385,0.90614,0.000594,"""normal"""
"""PE""",81,-140.417741,-128.507608,0.019735,0.127473,0.001864,0.693754,0.306246,1.0,4.1873e9,"""normal"""
"""NPI""",81,-196.878115,-184.967982,0.019838,-0.022922,0.000952,1.0,0.0,1.0,NaN,"""normal"""


In [6]:
def fit_rvine_copula(marginals: list[dict]) -> dict:
    u = np.column_stack([fit['pit'] for fit in marginals])
    controls = pv.FitControlsVinecop(
        family_set=[pv.student, pv.clayton, pv.gumbel, pv.frank, pv.gaussian],
        selection_criterion='bic',
        preselect_families=True,
        tree_criterion='tau',
    )
    copula = pv.Vinecop.from_data(data=u, controls=controls)

    copula_rows = []
    for tree_idx, family_row in enumerate(copula.families, start=1):
        tau_row = copula.taus[tree_idx - 1]
        for edge_idx, family in enumerate(family_row, start=1):
            copula_rows.append({
                'Tree': tree_idx,
                'Edge': edge_idx,
                'Family': str(family),
                'Tau': float(tau_row[edge_idx - 1]),
            })

    return {
        'copula': copula,
        'u': u,
        'summary': pl.DataFrame(copula_rows),
    }


vine_result = fit_rvine_copula(marginals)
rvine_copula = vine_result['copula']
copula_summary = vine_result['summary']
copula_summary

Tree,Edge,Family,Tau
i64,i64,str,f64
1,1,"""BicopFamily.gaussian""",0.310126
1,2,"""BicopFamily.clayton""",0.030649
1,3,"""BicopFamily.gumbel""",0.233544
2,1,"""BicopFamily.clayton""",0.080297
2,2,"""BicopFamily.gumbel""",0.041768
3,1,"""BicopFamily.clayton""",-0.106622


In [7]:
rvine_copula

<pyvinecopulib.Vinecop> Vinecop model with 4 variables
tree edge conditioned variables conditioning variables var_types   family rotation parameters  df   tau 
   1    1                  1, 3                             c, c Gaussian        0       0.47 1.0  0.31 
   1    2                  2, 4                             c, c  Clayton        0       0.06 1.0  0.03 
   1    3                  3, 4                             c, c   Gumbel        0       1.30 1.0  0.23 
   2    1                  1, 4                      3      c, c  Clayton      180       0.17 1.0  0.08 
   2    2                  2, 3                      4      c, c   Gumbel      180       1.04 1.0  0.04 
   3    1                  1, 2                   4, 3      c, c  Clayton      270       0.24 1.0 -0.11 

In [19]:
z = marginals[0]['ppf'](u[:, 0])


# plot time series of z
df = pd.DataFrame({'value': z}).reset_index()

# 3. Create the Chart
chart = alt.Chart(df).mark_line().encode(
    x=alt.X('index:Q', title='Time (Integer Index)'), # :Q specifies Quantitative data
    y=alt.Y('value:Q', title='Magnitude'),
    tooltip=['index', 'value'] # Adds interactivity
).properties(
    title='z',
    width=600,
    height=300
).interactive()

chart.display()

alt.Chart(...)

In [109]:
prev_return = np.column_stack([np.full(1, fit['last_return'], dtype=float) for fit in marginals])
prev_eps = np.column_stack([np.full(1, fit['last_sigma'] * fit['last_std_resid'], dtype=float) for fit in marginals])
prev_sigma = np.column_stack([np.full(1, max(fit['last_sigma'], 1e-8), dtype=float) for fit in marginals])

horizon_quarters = 300

returns = np.zeros((1, horizon_quarters, 4), dtype=float)
sigmas = np.zeros((1, horizon_quarters, 4), dtype=float)
zs = np.zeros((1, horizon_quarters, 4), dtype=float)


for quarter in range(horizon_quarters):
    u = rvine_copula.simulate(1)
    # print(f'Quarter {quarter + 1}/{horizon_quarters}, simulated copula u shape: {u.shape}, values: {u}')
    for idx, fit in enumerate(marginals):
        # prev_eps[:, idx] = np.clip(prev_eps[:, idx], -0.5, 0.5)
        z = fit['ppf'](u[:, idx])
        # sigma2 = 0.005 + 0.1 * np.square(prev_eps[:, idx]) + 0.8 * np.square(prev_sigma[:, idx])
        sigma2 = fit['omega'] + fit['alpha'] * np.square(prev_eps[:, idx]) + fit['beta'] * np.square(prev_sigma[:, idx])
        sigma = np.sqrt(np.maximum(sigma2, 1e-12))
        mean = fit['const'] + fit['phi'] * prev_return[:, idx] # EXPECTED_RETURNS[fit['asset']]
        eps = sigma * z
        r = mean + eps
        r_pct = np.exp(r) - 1.0
        # r = np.clip(r, -float('inf'), 2.0)
        returns[:, quarter, idx] = r_pct
        prev_return[:, idx] = r
        prev_eps[:, idx] = eps
        prev_sigma[:, idx] = sigma
        sigmas[:, quarter, idx] = sigma
        zs[:, quarter, idx] = z
        
df = pl.DataFrame({'returns': returns[:, :, 0].flatten(), 'sigmas': sigmas[:, :, 0].flatten()})

In [96]:
def plot_polars_timeseries(df: pl.DataFrame, x_col: str = None):
    """
    Plots all numeric columns of a Polars DataFrame as a time series.
    
    Args:
        df: The Polars DataFrame.
        x_col: The column to use for the x-axis. If None, an integer index is created.
    """
    # 1. Ensure an x-axis exists
    if x_col is None:
        df = df.with_row_index(name="index")
        x_col = "index"

    # 2. Reshape from Wide to Long format
    # This puts all values in one column and variable names in another
    value_vars = [c for c in df.columns if c != x_col]
    
    df_long = df.unpivot(
        index=x_col,
        on=value_vars,
        variable_name="column_name",
        value_name="reading"
    )

    # 3. Build the Altair Chart
    chart = alt.Chart(df_long.to_pandas()).mark_line().encode(
        x=alt.X(f"{x_col}:Q", title="Time Index"),
        y=alt.Y("reading:Q", title="Value"),
        color=alt.Color("column_name:N", title="Series"),
        tooltip=[x_col, "column_name", "reading"]
    ).properties(
        width=700,
        height=400,
        title="Multi-Column Time Series"
    ).interactive()

    return chart

In [110]:
plot_polars_timeseries(df)

alt.Chart(...)

In [ ]:
def simulate_3y_paths(marginals: list[dict], copula, n_simulations: int, horizon_quarters: int, seed: int = 7) -> dict:
    d = len(marginals)
    returns = np.zeros((n_simulations, horizon_quarters, d), dtype=float)

    prev_return = np.column_stack([np.full(n_simulations, fit['last_return'], dtype=float) for fit in marginals])
    prev_eps = np.column_stack([np.full(n_simulations, fit['last_sigma'] * fit['last_std_resid'], dtype=float) for fit in marginals])
    prev_sigma = np.column_stack([np.full(n_simulations, max(fit['last_sigma'], 1e-8), dtype=float) for fit in marginals])

    for quarter in range(horizon_quarters):
        u = copula.simulate(n_simulations, seeds=[seed + quarter])
        # print(f'Quarter {quarter + 1}/{horizon_quarters}, simulated copula u shape: {u.shape}, values: {u}')
        for idx, fit in enumerate(marginals):
            # prev_eps[:, idx] = np.clip(prev_eps[:, idx], -0.5, 0.5)
            z = fit['ppf'](u[:, idx])
            sigma2 = fit['omega'] + 0.5*fit['alpha'] * np.square(prev_eps[:, idx]) + 0.5*fit['beta'] * np.square(prev_sigma[:, idx])
            sigma = np.sqrt(np.maximum(sigma2, 1e-12))
            mean = fit['const'] + fit['phi'] * prev_return[:, idx] # EXPECTED_RETURNS[fit['asset']]
            eps = sigma * z
            r = mean + eps
            r_pct = np.exp(r) - 1.0
            print(f'Quarter {quarter + 1}/{horizon_quarters}, Asset {idx}, Max Return: {np.max(r_pct)}, Min Return: {np.min(r_pct)}')
            print(f'Quarter {quarter + 1}/{horizon_quarters}, Asset {idx}, Max Mean: {np.max(mean)}, Min Mean: {np.min(mean)}')
            print(f'Quarter {quarter + 1}/{horizon_quarters}, Asset {idx}, Max eps: {np.max(eps)}, Min eps: {np.min(eps)}')
            print(f'Quarter {quarter + 1}/{horizon_quarters}, Asset {idx}, Max Sigma: {np.max(sigma)}, Min Sigma: {np.min(sigma)}')
            print(f'Quarter {quarter + 1}/{horizon_quarters}, Asset {idx}, Max Z: {np.max(z)}, Min Z: {np.min(z)}')
            # r = np.clip(r, -float('inf'), 2.0)
            returns[:, quarter, idx] = r_pct
            prev_return[:, idx] = r
            prev_eps[:, idx] = eps
            prev_sigma[:, idx] = sigma

    weights = np.array([PORTFOLIO_WEIGHTS[fit['asset']] for fit in marginals], dtype=float)
    quarter_portfolio_pct = np.einsum('qij,j->qi', returns, weights)
    horizon_compounded_return = np.prod(1.0 + quarter_portfolio_pct, axis=1) - 1.0
    horizon_asset_linear = np.sum(returns * weights.reshape(1, 1, -1), axis=1)
    

    return {
        'returns': returns,
        'weights': weights,
        'quarter_portfolio': quarter_portfolio_pct,
        'compounded_return': horizon_compounded_return,
        'asset_contrib_simple': horizon_asset_linear,
    }


simulation = simulate_3y_paths(marginals, rvine_copula, N_SIMULATIONS, HORIZON_QUARTERS, seed=RNG_SEED)
simulation_pl = pl.DataFrame({
    'compounded_return': simulation['compounded_return'],
})
simulation_pl.head()

In [ ]:
q_5 = simulation_pl.select(pl.col('compounded_return').quantile(ALPHA)).item()
cvar_95 = simulation_pl.filter(pl.col('compounded_return') <= q_5).select(pl.col('compounded_return').mean()).item()
var_95 = q_5
mean_return = simulation_pl.select(pl.col('compounded_return').mean()).item()
std_return = simulation_pl.select(pl.col('compounded_return').std()).item()
tail_mean = simulation_pl.filter(pl.col('compounded_return') <= q_5).select(pl.col('compounded_return').mean()).item()

component_rows = []
tail_mask = simulation['compounded_return'] <= q_5
for idx, fit in enumerate(marginals):
    component = -simulation['asset_contrib_simple'][tail_mask, idx].mean()
    component_rows.append({
        'Asset': fit['asset'],
        'ComponentCVaR': float(component),
        'ShareOfCVaR': float(component / cvar_95),
    })
component_summary = pl.DataFrame(component_rows)

risk_summary = pl.DataFrame([
    {'Metric': 'Mean 3Y Return', 'Value': mean_return},
    {'Metric': 'Std Dev', 'Value': std_return},
    {'Metric': 'VaR 95%', 'Value': var_95},
    {'Metric': 'CVaR 95%', 'Value': cvar_95},
])

risk_summary

In [ ]:
asset_chart = (
    alt.Chart(pd.DataFrame(returns_long.to_dict(as_series=False)))
    .mark_line()
    .encode(
        x=alt.X(f'{DATE_COLUMN}:T', title='Date'),
        y=alt.Y('return:Q', title='Quarterly return'),
        color=alt.Color('asset:N', title='Asset'),
    )
    .properties(title='Quarterly Returns by Asset', width=820, height=320)
)

distribution_chart = (
    alt.Chart(pd.DataFrame(simulation_pl.to_dict(as_series=False)))
    .transform_bin('bin', 'compounded_return', bin=alt.Bin(maxbins=150))
    .transform_aggregate(count='count()', groupby=['bin'])
    .mark_bar(color='#2a9d8f')
    .encode(
        x=alt.X('bin:Q', title='3-year compounded return'),
        y=alt.Y('count:Q', title='Frequency'),
    )
    .properties(title='Simulated 3-Year Portfolio Return Distribution', width=820, height=300)
)

tail_rule = alt.Chart(pd.DataFrame({'x': [q_5], 'label': ['VaR 5%']})).mark_rule(color='crimson', strokeDash=[6, 4]).encode(x='x:Q')
cvar_rule = alt.Chart(pd.DataFrame({'x': [tail_mean], 'label': ['Tail Mean']})).mark_rule(color='black').encode(x='x:Q')

risk_bar = (
    alt.Chart(pd.DataFrame(component_summary.to_dict(as_series=False)))
    .mark_bar()
    .encode(
        x=alt.X('Asset:N', title='Asset'),
        y=alt.Y('ComponentCVaR:Q', title='CVaR contribution'),
        color=alt.Color('Asset:N', legend=None),
        tooltip=['Asset', 'ComponentCVaR', 'ShareOfCVaR'],
    )
    .properties(title='Component CVaR Contributions', width=520, height=300)
)

asset_chart

In [ ]:
distribution_chart_with_rules = distribution_chart + tail_rule + cvar_rule
distribution_chart_with_rules

In [ ]:
risk_bar

In [ ]:
simulation_pl = simulation_pl.with_columns(pl.col('compounded_return').log().alias('log_return'))

In [ ]:
simulation_pl

In [ ]:
distribution_chart = (
    alt.Chart(pd.DataFrame(simulation_pl.to_dict(as_series=False)))
    .transform_bin('bin', 'log_return', bin=alt.Bin(maxbins=150))
    .transform_aggregate(count='count()', groupby=['bin'])
    .mark_bar(color='#2a9d8f')
    .encode(
        x=alt.X('bin:Q', title='3-year compounded return'),
        y=alt.Y('count:Q', title='Frequency'),
    )
    .properties(title='Simulated 3-Year Portfolio Return Distribution', width=820, height=300)
)
distribution_chart

## Notes
- The notebook keeps the provided CSV load line intact and does not modify any existing project file.
- `RESIDUAL_FAMILY` can be switched between `skewt` and `stable`.
- If you want to fit on appraisal-smoothed private returns instead of the already-processed dataset, set `APPLY_UNSMOOTHING = True`.
- The CVaR decomposition below uses the linearized portfolio return in decimal units, while the reported portfolio CVaR is based on the 3-year compounded return.